
# GenAI Evaluation — End-to-End Practical Notebook

### Employee Policy RAG + LLM Evaluation + Agent Evaluation + Production Metrics

This notebook converts the complete practical into a clean, classroom-friendly workflow.

## What you will build

```text
User Query
    ↓
Retriever
    ↓
Retrieval Evaluation
    ├── Precision@K
    ├── Recall@K
    ├── Hit Rate
    ├── Reciprocal Rank / MRR
    └── NDCG
    ↓
RAG Generation
    ↓
LLM-as-a-Judge
    ├── Correctness
    ├── Relevance
    ├── Faithfulness
    └── Instruction Following
    ↓
Agent Evaluation
    ├── Tool Selection
    ├── Tool Calls
    ├── Trajectory
    └── Step Efficiency
    ↓
Production Evaluation
    ├── Latency
    ├── Token Usage
    ├── Error Rate
    └── User Feedback
```

> **Recommended:** Run the notebook top-to-bottom. OpenAI-dependent cells require an `OPENAI_API_KEY`.



## 0. Installation

Run this once in a fresh notebook environment.


In [ ]:
%pip install -U langchain langchain-openai pydantic pandas numpy


## 1. Imports and Configuration

We keep the model names configurable through environment variables.

- `OPENAI_MODEL` defaults to `gpt-5.5`
- `OPENAI_EMBEDDING_MODEL` defaults to `text-embedding-3-small`


In [1]:
import os
import math
import time
import getpass
import pandas as pd

from typing import List
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

print("Imports successful.")

d:\complete_content_new\Full-Stack-GenAI-Bootcamp-1.0\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports successful.



### Configure the API key

The key is requested interactively only when it is not already available in the environment.


In [2]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI_API_KEY: ")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
OPENAI_EMBEDDING_MODEL = os.getenv(
    "OPENAI_EMBEDDING_MODEL",
    "text-embedding-3-small"
)

print("Chat model:", OPENAI_MODEL)
print("Embedding model:", OPENAI_EMBEDDING_MODEL)

Chat model: gpt-5.5
Embedding model: text-embedding-3-small


In [3]:
llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model=OPENAI_EMBEDDING_MODEL
)

print("LLM and embedding objects created.")

LLM and embedding objects created.



# 2. Mini Knowledge Base

We will evaluate a small HR-policy RAG system.


In [4]:
documents = [
    Document(
        page_content="""
        Employees are allowed to work from home
        for a maximum of 2 days per week.
        """,
        metadata={"doc_id": "remote_policy"}
    ),

    Document(
        page_content="""
        Every full-time employee receives
        24 paid leaves per calendar year.
        """,
        metadata={"doc_id": "leave_policy"}
    ),

    Document(
        page_content="""
        Employees can claim up to ₹3000 per month
        for internet reimbursement.
        """,
        metadata={"doc_id": "internet_policy"}
    ),

    Document(
        page_content="""
        The standard probation period
        for new employees is 6 months.
        """,
        metadata={"doc_id": "probation_policy"}
    ),

    Document(
        page_content="""
        Employees receive ₹1000 per month
        as mobile reimbursement.
        """,
        metadata={"doc_id": "mobile_policy"}
    )
]

print(f"Loaded {len(documents)} policy documents.")
for doc in documents:
    print("-", doc.metadata["doc_id"])

Loaded 5 policy documents.
- remote_policy
- leave_policy
- internet_policy
- probation_policy
- mobile_policy



# 3. Create the Vector Store


In [5]:
vector_store = InMemoryVectorStore(
    embedding=embeddings
)

ids = [
    doc.metadata["doc_id"]
    for doc in documents
]

vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Documents indexed successfully.")

Documents indexed successfully.



# 4. Retriever

The retriever returns the top-`k` semantically similar documents.


In [6]:
def retrieve(query: str, k: int = 3):
    docs = vector_store.similarity_search(
        query,
        k=k
    )
    return docs

In [7]:
results = retrieve(
    "How many paid leaves do employees receive?"
)

for rank, doc in enumerate(results, start=1):
    print(f"Rank {rank}: {doc.metadata['doc_id']}")
    print(doc.page_content.strip())
    print("-" * 60)

Rank 1: leave_policy
Every full-time employee receives
        24 paid leaves per calendar year.
------------------------------------------------------------
Rank 2: remote_policy
Employees are allowed to work from home
        for a maximum of 2 days per week.
------------------------------------------------------------
Rank 3: mobile_policy
Employees receive ₹1000 per month
        as mobile reimbursement.
------------------------------------------------------------



# PART 1 — Manual Retrieval Evaluation

For a query, we need:

- **Retrieved documents** — what our retriever returned.
- **Relevant documents / ground truth** — what should have been returned.

Example:

```text
Relevant document: leave_policy

Top-3 retrieved:
1. leave_policy
2. probation_policy
3. remote_policy
```


In [8]:
relevant_docs = ["leave_policy"]

retrieved_docs = [
    "leave_policy",
    "probation_policy",
    "remote_policy"
]

print("Relevant:", relevant_docs)
print("Retrieved:", retrieved_docs)

Relevant: ['leave_policy']
Retrieved: ['leave_policy', 'probation_policy', 'remote_policy']



## 5. Precision@K

\[
Precision@K = 
rac{	ext{Relevant documents retrieved in top K}}{K}
\]

For this example:

```text
Relevant in Top-3 = 1
K = 3

Precision@3 = 1 / 3 = 0.3333
```


In [9]:
def precision_at_k(
    retrieved: List[str],
    relevant: List[str],
    k: int
):
    retrieved_k = retrieved[:k]

    relevant_count = sum(
        doc in relevant
        for doc in retrieved_k
    )

    return relevant_count / k

In [10]:
score = precision_at_k(
    retrieved_docs,
    relevant_docs,
    k=3
)

print("Precision@3 =", round(score, 4))

Precision@3 = 0.3333



## 6. Recall@K

\[
Recall@K = 
rac{	ext{Relevant documents retrieved in top K}}
{	ext{Total relevant documents}}
\]

Here the only relevant document was retrieved, therefore:

```text
Recall@3 = 1 / 1 = 1.0
```


In [11]:
def recall_at_k(
    retrieved: List[str],
    relevant: List[str],
    k: int
):
    retrieved_k = retrieved[:k]

    found_relevant = sum(
        doc in retrieved_k
        for doc in relevant
    )

    return found_relevant / len(relevant)

In [12]:
recall = recall_at_k(
    retrieved_docs,
    relevant_docs,
    3
)

print("Recall@3 =", recall)

Recall@3 = 1.0



## 7. Hit Rate

Hit Rate answers one simple question:

> Did at least one relevant document appear in the Top-K?

```text
YES → 1
NO  → 0
```


In [13]:
def hit_rate(
    retrieved: List[str],
    relevant: List[str],
    k: int
):
    retrieved_k = retrieved[:k]

    hit = any(
        doc in relevant
        for doc in retrieved_k
    )

    return 1 if hit else 0

In [14]:
print(
    "Hit Rate =",
    hit_rate(
        retrieved_docs,
        relevant_docs,
        3
    )
)

Hit Rate = 1



## 8. Reciprocal Rank and MRR

For one query:

\[
RR = 
rac{1}{	ext{rank of first relevant result}}
\]

Across many queries:

\[
MRR = 
rac{1}{N}\sum_{i=1}^{N}RR_i
\]


In [15]:
def reciprocal_rank(
    retrieved: List[str],
    relevant: List[str]
):
    for rank, doc in enumerate(
        retrieved,
        start=1
    ):
        if doc in relevant:
            return 1 / rank

    return 0

In [16]:
case_1 = [
    "leave_policy",
    "remote_policy",
    "internet_policy"
]

case_2 = [
    "remote_policy",
    "internet_policy",
    "leave_policy"
]

print(
    "RR when relevant doc is rank 1:",
    reciprocal_rank(case_1, ["leave_policy"])
)

print(
    "RR when relevant doc is rank 3:",
    round(reciprocal_rank(case_2, ["leave_policy"]), 4)
)

RR when relevant doc is rank 1: 1.0
RR when relevant doc is rank 3: 0.3333



## 9. NDCG@K

NDCG evaluates **ranking quality**, not just whether the relevant item was retrieved.

A relevant document at Rank 1 should receive more credit than the same document at Rank 3.


In [17]:
def ndcg_at_k(
    retrieved: List[str],
    relevant: List[str],
    k: int
):
    retrieved_k = retrieved[:k]

    dcg = 0

    for i, doc in enumerate(
        retrieved_k,
        start=1
    ):
        relevance = (
            1 if doc in relevant
            else 0
        )

        dcg += relevance / math.log2(i + 1)

    ideal_relevant_count = min(
        len(relevant),
        k
    )

    idcg = sum(
        1 / math.log2(i + 1)
        for i in range(
            1,
            ideal_relevant_count + 1
        )
    )

    if idcg == 0:
        return 0

    return dcg / idcg

In [18]:
score = ndcg_at_k(
    [
        "remote_policy",
        "leave_policy",
        "internet_policy"
    ],
    ["leave_policy"],
    3
)

print("NDCG@3 =", round(score, 4))

NDCG@3 = 0.6309



### Important Observation

A retriever can have:

```text
Recall@3 = 1.0
NDCG@3   < 1.0
```

That means the relevant document was found, but its rank was not ideal.



## 10. Sanity Tests for Manual Metrics

These tests do **not** call an LLM or embedding API. They verify the manual metric implementations.


In [19]:
assert abs(
    precision_at_k(
        ["leave_policy", "remote_policy", "internet_policy"],
        ["leave_policy"],
        3
    ) - (1 / 3)
) < 1e-9

assert recall_at_k(
    ["leave_policy", "remote_policy", "internet_policy"],
    ["leave_policy"],
    3
) == 1.0

assert hit_rate(
    ["leave_policy", "remote_policy"],
    ["leave_policy"],
    2
) == 1

assert reciprocal_rank(
    ["remote_policy", "internet_policy", "leave_policy"],
    ["leave_policy"]
) == (1 / 3)

assert abs(
    ndcg_at_k(
        ["remote_policy", "leave_policy", "internet_policy"],
        ["leave_policy"],
        3
    ) - (1 / math.log2(3))
) < 1e-9

print("✅ All manual metric tests passed.")

✅ All manual metric tests passed.



# PART 2 — End-to-End RAG Evaluation

Now we connect the retriever with the LLM.


In [20]:
def rag_answer(query: str, k: int = 3):
    retrieved_docs = retrieve(
        query,
        k=k
    )

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_docs
    )

    prompt = f"""
You are an HR policy assistant.

Answer the question using ONLY
the provided context.

If the answer is not available in
the context, say:

"I don't know based on the provided context."

CONTEXT:
{context}

QUESTION:
{query}
"""

    start_time = time.perf_counter()

    response = llm.invoke(prompt)

    latency = (
        time.perf_counter()
        - start_time
    )

    return {
        "answer": response.content,
        "retrieved_docs": retrieved_docs,
        "latency": latency,
        "response_metadata": response.response_metadata,
        "usage_metadata": response.usage_metadata or {}
    }

In [21]:
result = rag_answer(
    "How many paid leaves do employees receive?"
)

print("Answer:")
print(result["answer"])

print("\nRetrieved documents:")
for doc in result["retrieved_docs"]:
    print("-", doc.metadata["doc_id"])

print("\nLatency:", round(result["latency"], 3), "seconds")
print("Token usage:", result["usage_metadata"])

Answer:
Every full-time employee receives 24 paid leaves per calendar year.

Retrieved documents:
- leave_policy
- remote_policy
- mobile_policy

Latency: 3.461 seconds
Token usage: {'input_tokens': 114, 'output_tokens': 16, 'total_tokens': 130, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}



# PART 3 — LLM-as-a-Judge

The evaluator will score four dimensions from **1 to 5**:

1. Correctness
2. Relevance
3. Faithfulness
4. Instruction Following


In [22]:
class EvaluationScore(BaseModel):
    correctness: int = Field(
        ge=1,
        le=5,
        description="Correctness score from 1 to 5"
    )

    relevance: int = Field(
        ge=1,
        le=5,
        description="Relevance score from 1 to 5"
    )

    faithfulness: int = Field(
        ge=1,
        le=5,
        description="Faithfulness to supplied context from 1 to 5"
    )

    instruction_following: int = Field(
        ge=1,
        le=5,
        description="Instruction-following score from 1 to 5"
    )

    explanation: str = Field(
        description="Short justification for the scores"
    )


We use OpenAI-native structured output through LangChain's `with_structured_output`.


In [23]:
judge = llm.with_structured_output(
    EvaluationScore,
    method="json_schema"
)

In [24]:
def evaluate_answer(
    question,
    answer,
    reference_answer,
    context
):
    judge_prompt = f"""
You are evaluating an AI system.

QUESTION:
{question}

REFERENCE ANSWER:
{reference_answer}

RETRIEVED CONTEXT:
{context}

AI ANSWER:
{answer}

Score the answer from 1 to 5
for the following:

Correctness:
Does the answer match the
reference answer?

Relevance:
Does the answer directly answer
the question?

Faithfulness:
Are all factual claims supported
by the retrieved context?

Instruction Following:
Did the assistant follow the
instruction to only use context?

Give a concise explanation.
"""

    return judge.invoke(
        judge_prompt
    )


# 11. Evaluation Dataset

A real evaluation needs a dataset, not a single prompt.


In [25]:
eval_dataset = [
    {
        "question":
            "How many paid leaves do employees receive?",

        "expected_answer":
            "24 paid leaves per calendar year.",

        "relevant_docs":
            ["leave_policy"]
    },

    {
        "question":
            "How many work from home days are allowed?",

        "expected_answer":
            "Maximum 2 days per week.",

        "relevant_docs":
            ["remote_policy"]
    },

    {
        "question":
            "What is the monthly internet reimbursement limit?",

        "expected_answer":
            "₹3000 per month.",

        "relevant_docs":
            ["internet_policy"]
    },

    {
        "question":
            "How long is the probation period?",

        "expected_answer":
            "6 months.",

        "relevant_docs":
            ["probation_policy"]
    }
]

pd.DataFrame(eval_dataset)

,question,expected_answer,relevant_docs
0,How many paid leaves do employees receive?,24 paid leaves per calendar year.,[leave_policy]
1,How many work from home days are allowed?,Maximum 2 days per week.,[remote_policy]
2,What is the monthly internet reimbursement limit?,₹3000 per month.,[internet_policy]
3,How long is the probation period?,6 months.,[probation_policy]



# 12. Full RAG Evaluation Loop

For every test case we evaluate both:

### Retrieval layer
- Precision@3
- Recall@3
- Hit Rate
- Reciprocal Rank
- NDCG@3

### Generation layer
- Correctness
- Relevance
- Faithfulness
- Instruction Following
- Latency


In [26]:
evaluation_results = []

for sample in eval_dataset:
    query = sample["question"]

    result = rag_answer(
        query,
        k=3
    )

    answer = result["answer"]

    retrieved_documents = (
        result["retrieved_docs"]
    )

    retrieved_ids = [
        doc.metadata["doc_id"]
        for doc in retrieved_documents
    ]

    context = "\n".join(
        doc.page_content
        for doc in retrieved_documents
    )

    precision = precision_at_k(
        retrieved_ids,
        sample["relevant_docs"],
        3
    )

    recall = recall_at_k(
        retrieved_ids,
        sample["relevant_docs"],
        3
    )

    hit = hit_rate(
        retrieved_ids,
        sample["relevant_docs"],
        3
    )

    rr = reciprocal_rank(
        retrieved_ids,
        sample["relevant_docs"]
    )

    ndcg = ndcg_at_k(
        retrieved_ids,
        sample["relevant_docs"],
        3
    )

    judge_result = evaluate_answer(
        question=query,
        answer=answer,
        reference_answer=sample[
            "expected_answer"
        ],
        context=context
    )

    evaluation_results.append({
        "question":
            query,

        "answer":
            answer,

        "precision@3":
            precision,

        "recall@3":
            recall,

        "hit_rate":
            hit,

        "reciprocal_rank":
            rr,

        "ndcg@3":
            ndcg,

        "correctness":
            judge_result.correctness,

        "relevance":
            judge_result.relevance,

        "faithfulness":
            judge_result.faithfulness,

        "instruction_following":
            judge_result.instruction_following,

        "latency":
            result["latency"]
    })


# 13. Evaluation Report


In [27]:
df = pd.DataFrame(
    evaluation_results
)

df

,question,answer,precision@3,recall@3,hit_rate,reciprocal_rank,ndcg@3,correctness,relevance,faithfulness,instruction_following,latency
0,How many paid leaves do employees receive?,Full-time employees receive 24 paid leaves per...,0.333333,1.0,1,1.0,1.0,5,5,5,5,2.167246
1,How many work from home days are allowed?,Employees are allowed to work from home for a ...,0.333333,1.0,1,1.0,1.0,5,5,5,5,1.909380
2,What is the monthly internet reimbursement limit?,Employees can claim up to ₹3000 per month for ...,0.333333,1.0,1,1.0,1.0,5,5,5,5,1.717986
3,How long is the probation period?,The standard probation period for new employee...,0.333333,1.0,1,1.0,1.0,5,5,5,5,1.383276


In [28]:
metric_columns = [
    "precision@3",
    "recall@3",
    "hit_rate",
    "reciprocal_rank",
    "ndcg@3",
    "correctness",
    "relevance",
    "faithfulness",
    "instruction_following",
    "latency"
]

overall_report = (
    df[metric_columns]
    .mean()
    .to_frame("mean_score")
)

overall_report

,mean_score
precision@3,0.333333
recall@3,1.000000
hit_rate,1.000000
reciprocal_rank,1.000000
ndcg@3,1.000000
correctness,5.000000
relevance,5.000000
faithfulness,5.000000
instruction_following,5.000000
latency,1.794472



# PART 4 — Hallucination / Faithfulness Practical

Deliberately introduce one unsupported claim.

Context:

```text
Employees receive 24 paid leaves per year.
```

Generated answer:

```text
Employees receive 24 paid leaves and 10 sick leaves every year.
```

Two factual claims are present:

1. `24 paid leaves` → supported ✅
2. `10 sick leaves` → unsupported ❌

A simplified claim-level faithfulness score would therefore be:

\[

rac{1}{2} = 0.5
\]


In [29]:
context = """
Employees receive 24 paid leaves
per year.
"""

answer = """
Employees receive 24 paid leaves
and 10 sick leaves every year.
"""

supported_claims = 1
total_claims = 2

manual_faithfulness = (
    supported_claims
    / total_claims
)

print("Simplified faithfulness =", manual_faithfulness)

Simplified faithfulness = 0.5



# PART 5 — Agent Evaluation

The agent receives two tools:

```text
policy_search
multiply
```

Ideal trajectory:

```text
User
 ↓
policy_search
 ↓
24 leaves/year
 ↓
multiply(24, 3)
 ↓
72
 ↓
Final Answer
```


In [30]:
from langchain.agents import create_agent
from langchain.tools import tool

In [31]:
@tool
def policy_search(query: str) -> str:
    """
    Search company HR policies.
    """
    docs = retrieve(
        query,
        k=2
    )

    return "\n".join(
        doc.page_content
        for doc in docs
    )


@tool
def multiply(
    a: float,
    b: float
) -> float:
    """
    Multiply two numbers.
    """
    return a * b

In [32]:
agent = create_agent(
    model=llm,
    tools=[
        policy_search,
        multiply
    ],
    system_prompt="""
You are an HR assistant.

Use policy_search whenever
company policy information is needed.

Use multiply for multiplication.
"""
)

print("Agent created.")

Agent created.


In [33]:
query = """
Find the annual paid leave
entitlement and calculate how many
paid leaves an employee receives
over 3 years.
"""

agent_result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": query
        }
    ]
})

final_message = agent_result["messages"][-1]
print(final_message.content)

Full-time employees receive **24 paid leave days per calendar year**.

Over **3 years**: **24 × 3 = 72 paid leave days**.



## 14. Extract Tool Calls


In [34]:
tool_calls = []

for message in agent_result["messages"]:
    if hasattr(
        message,
        "tool_calls"
    ):
        if message.tool_calls:
            for call in message.tool_calls:
                tool_calls.append(
                    call["name"]
                )

print("Tool calls:", tool_calls)

Tool calls: ['policy_search', 'multiply']



## 15. Tool Selection Accuracy


In [35]:
expected_tools = [
    "policy_search",
    "multiply"
]

actual_tools = tool_calls


def tool_selection_accuracy(
    expected,
    actual
):
    correct = sum(
        tool in actual
        for tool in expected
    )

    return correct / len(expected)


tool_accuracy = tool_selection_accuracy(
    expected_tools,
    actual_tools
)

print("Tool Selection Accuracy =", tool_accuracy)

Tool Selection Accuracy = 1.0



## 16. Agent Step Efficiency

If the ideal trajectory requires `2` tool calls but the agent makes `5`, then:

\[
Efficiency = 
rac{2}{5} = 0.4
\]


In [36]:
number_of_steps = len(
    tool_calls
)


def step_efficiency(
    expected_steps,
    actual_steps
):
    if actual_steps <= expected_steps:
        return 1.0

    return expected_steps / actual_steps


print(
    "Example efficiency for 2 expected vs 5 actual:",
    step_efficiency(
        2,
        5
    )
)

print(
    "Efficiency for this agent run:",
    step_efficiency(
        2,
        number_of_steps
    )
)

Example efficiency for 2 expected vs 5 actual: 0.4
Efficiency for this agent run: 1.0



# PART 6 — Production Evaluation

In production we care about system behavior in addition to answer quality.

Useful signals include:

- P50 / P95 / P99 latency
- Input and output tokens
- Cost per request
- Error rate
- User feedback
- Faithfulness trend
- Task success rate


In [37]:
production_metrics = {
    "latency_seconds":
        result["latency"],

    "input_tokens":
        result["usage_metadata"]
        .get("input_tokens"),

    "output_tokens":
        result["usage_metadata"]
        .get("output_tokens"),

    "total_tokens":
        result["usage_metadata"]
        .get("total_tokens"),

    "success":
        True,

    "user_feedback":
        None
}

production_metrics

{'latency_seconds': 1.3832755000330508,
 'input_tokens': 114,
 'output_tokens': 15,
 'total_tokens': 129,
 'success': True,
 'user_feedback': None}


### Latency Example


In [38]:
latencies = [
    1.2,
    1.4,
    2.1,
    1.8,
    5.4
]

average_latency = (
    sum(latencies)
    / len(latencies)
)

print("Average latency =", round(average_latency, 3), "seconds")

Average latency = 2.38 seconds



## 17. Percentile Latency

For production systems, averages can hide slow outliers, so percentile latency is useful.


In [39]:
import numpy as np

print("P50:", round(float(np.percentile(latencies, 50)), 3))
print("P95:", round(float(np.percentile(latencies, 95)), 3))
print("P99:", round(float(np.percentile(latencies, 99)), 3))

P50: 1.8
P95: 4.74
P99: 5.268



# Final Architecture

```text
                  USER QUERY
                      │
                      ▼
                 RETRIEVER
                      │
          ┌───────────┴───────────┐
          │                       │
          ▼                       ▼
 Retrieval Evaluation        Retrieved Context
                              │
 Precision@K                  ▼
 Recall@K                    LLM
 Hit Rate                     │
 MRR                          ▼
 NDCG                    Generated Answer
                              │
                              ▼
                     Generation Evaluation
                              │
                         Correctness
                         Relevance
                         Faithfulness
                         Instruction Following


AGENT
  │
  ├── Tool Selection
  ├── Tool Call Accuracy
  ├── Trajectory
  ├── Task Success
  └── Step Efficiency


PRODUCTION
  │
  ├── Latency
  ├── Tokens
  ├── Cost
  ├── Error Rate
  ├── Safety
  └── User Feedback
```



# Validation Notes

This notebook was prepared with the following checks:

- ✅ All ordinary Python code cells were syntax-checked.
- ✅ Manual metric implementations were independently tested.
- ✅ Expected values verified:
  - Precision@3 = `0.3333`
  - Recall@3 = `1.0`
  - Hit Rate = `1`
  - Reciprocal Rank at rank 3 = `0.3333`
  - NDCG for one relevant result at rank 2 = `0.6309`
  - Simplified faithfulness example = `0.5`
  - Step efficiency for 2 expected / 5 actual = `0.4`
- ✅ Current LangChain structure used for:
  - `ChatOpenAI`
  - `OpenAIEmbeddings`
  - `InMemoryVectorStore`
  - `with_structured_output`
  - `create_agent`
  - `@tool`
- ✅ Token accounting uses `AIMessage.usage_metadata` in the production section.
- ⚠️ Cells that invoke OpenAI or embeddings require a valid `OPENAI_API_KEY` and network access.

## Suggested teaching flow

```text
Manual Retrieval Metrics
        ↓
RAG Pipeline
        ↓
LLM-as-a-Judge
        ↓
Full Evaluation Dataset
        ↓
Hallucination / Faithfulness
        ↓
Agent Evaluation
        ↓
Production Metrics
```
